In [0]:
stream_path = "/Volumes/ev-charge/default/ev-data/ev_station_streaming"
checkpoint_path = "/Volumes/ev-charge/default/ev-data/ev_station_checkpoint"

dbutils.fs.mkdirs(stream_path)
dbutils.fs.mkdirs(checkpoint_path)

print("Streaming folders created")

Streaming folders created


In [0]:
events = [
    {
        "station_id": "EV001",
        "vehicle_id": "V001",
        "event_type": "charging_started",
        "power_kw": 40.5,
        "energy_kwh": 10.2,
        "event_ts": "2026-10-06T10:00:00"
    },
    {
        "station_id": "EV002",
        "vehicle_id": "V002",
        "event_type": "charging_started",
        "power_kw": 35.0,
        "energy_kwh": 8.5,
        "event_ts": "2026-10-06T10:02:00"
    }
]

dbutils.fs.put(
    stream_path + "/events_01.json",
    "\n".join([str(x).replace("'", '"') for x in events]),
    True
)

print("Initial EV events created")

Wrote 304 bytes.
Initial EV events created


In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("station_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("power_kw", DoubleType(), True),
    StructField("energy_kwh", DoubleType(), True),
    StructField("event_ts", StringType(), True)
])

stream_df = (
    spark.readStream
    .schema(schema)
    .json(stream_path)
)

query = (
    stream_df.writeStream
    .outputMode("append")
    .trigger(availableNow=True)
    .format("memory")
    .option("checkpointLocation", checkpoint_path + "_mem")
    .queryName("ev_stream")
    .start()
)

query.awaitTermination()

display(spark.table("ev_stream"))

station_id,vehicle_id,event_type,power_kw,energy_kwh,event_ts
EV001,V001,charging_started,40.5,10.2,2026-10-06T10:00:00
EV002,V002,charging_started,35.0,8.5,2026-10-06T10:02:00


In [0]:
bronze_stream = (
    stream_df
    .withColumn("ingestion_time", current_timestamp())
)

query = (
    bronze_stream.writeStream
    .trigger(availableNow=True)
    .format("memory")
    .queryName("ev_station_bronze")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path + "_bronze")
    .start()
)

print("Streaming query started")

Streaming query started


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ev_station_bronze
    """)
)

station_id,vehicle_id,event_type,power_kw,energy_kwh,event_ts,ingestion_time
EV001,V001,charging_started,40.5,10.2,2026-10-06T10:00:00,2026-10-06T17:18:51.242Z
EV002,V002,charging_started,35.0,8.5,2026-10-06T10:02:00,2026-10-06T17:18:51.242Z


In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_events,
            SUM(energy_kwh) AS total_energy_kwh
        FROM ev_station_bronze
    """)
)

total_events,total_energy_kwh
2,18.7


In [0]:
new_event = {
    "station_id": "EV003",
    "vehicle_id": "V003",
    "event_type": "charging_started",
    "power_kw": 45.0,
    "energy_kwh": 15.5,
    "event_ts": "2026-10-06T10:05:00"
}

dbutils.fs.put(
    stream_path + "/events_02.json",
    str(new_event).replace("'", '"'),
    True
)

print("New EV charging event added")

Wrote 152 bytes.
New EV charging event added


In [0]:
# Read all events from the streaming source (picks up newly added files)
display(
    spark.read.json(stream_path)
        .withColumn("ingestion_time", current_timestamp())
)

energy_kwh,event_ts,event_type,power_kw,station_id,vehicle_id,ingestion_time
10.2,2026-10-06T10:00:00,charging_started,40.5,EV001,V001,2026-10-06T17:33:16.345Z
8.5,2026-10-06T10:02:00,charging_started,35.0,EV002,V002,2026-10-06T17:33:16.345Z
15.5,2026-10-06T10:05:00,charging_started,45.0,EV003,V003,2026-10-06T17:33:16.345Z


In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_events,
            SUM(energy_kwh) AS total_energy_kwh
        FROM ev_station_bronze
    """)
)

total_events,total_energy_kwh
3,34.2


In [0]:
query.stop()

print("Streaming query stopped")

Streaming query stopped
